# Train Llama 70B

In [ ]:
from unsloth import FastLanguageModel
import torch

base_model = "meta-llama/Llama-3.1-70B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    max_seq_length = 1024,
    load_in_4bit = True,         
    dtype = torch.bfloat16,       
    trust_remote_code = True,
    device_map = "auto",          
)

model.config.use_cache = False

# Only if generation_config exists
if hasattr(model, "generation_config"):
    model.generation_config.use_cache = False

print(f"Loaded model {base_model} in 4-bit ✅")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/59.6k [00:00<?, ?B/s]

model-00001-of-00030.safetensors:   0%|          | 0.00/4.58G [00:00<?, ?B/s]

model-00002-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

model-00003-of-00030.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00030.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00005-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

model-00006-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

model-00007-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

model-00008-of-00030.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

# Apply LoRa adapter

In [4]:
peft_model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Keep: balances capacity and efficiency
    lora_alpha=48,  # Lowered: reduces overfitting risk (scale = 1)
    lora_dropout=0.10,  # Slightly higher: better regularization for noisy esoteric texts
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention: core for reasoning
        "gate_proj"
        #"up_proj", "down_proj", "gate_proj"  # MLP: adds expressivity for complex patterns
    ],
    bias="none",  # Keep: minimizes params
    use_gradient_checkpointing=True,  # Keep: VRAM saver
    modules_to_save=None,  # Default: avoids retraining embeddings
    use_rslora=False  # NEW: rank-stabilized LoRA, improves stability for higher r
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

for cfg in [model.config]:
    cfg.bos_token_id = tokenizer.bos_token_id
    cfg.eos_token_id = tokenizer.eos_token_id
    cfg.pad_token_id = tokenizer.pad_token_id
    
peft_model.config.use_cache = False

print("Loaded peft model ✅")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.1.4 patched 80 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Loaded peft model ✅


# Load the dataset from corpus

In [5]:
import os 

CORPUS_DIR = "/storage/corpus/corpus_gamma_mini/"

BLOCK_SIZE = 1024  # max tokens per chunk

tok = tokenizer 

# Ensure EOS/PAD exist and are consistent
added = False
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": "</s>"})
    added = True
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    added = True
if added:
    model.resize_token_embeddings(len(tok))

# -----------------------
# 2) Load raw text files (no EOS strings here)
# -----------------------
def load_txt_corpus(directory):
    texts = []
    for filename in os.listdir(directory):
        if not filename.endswith(".txt"):
            continue
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read().strip()
            if txt:
                texts.append(txt)
    return texts

raw_texts = load_txt_corpus(CORPUS_DIR)
print(f"Loaded {len(raw_texts)} files from {CORPUS_DIR} ✅")

Loaded 11216 files from /storage/corpus/corpus_gamma_mini/ ✅


# Tokenize with EOS appended (token id, not string)
We’ll build one long stream of ids and then pack into BLOCK_SIZE chunks.

In [6]:
from datasets import Dataset

# -------------------------------------------------
# 1. Tokenize + append EOS (one token per example)
# -------------------------------------------------
def tokenize_append_eos(texts):
    """
    texts: str   OR   list[str]
    Returns: flat list[int] of token ids with an EOS after every example.
    """
    # ------------------------------------------------------------------
    # 1. Make sure we always have a list of strings
    # ------------------------------------------------------------------
    if isinstance(texts, str):
        texts = [texts]                     # single example → batch of 1
    elif not isinstance(texts, list):
        raise TypeError("`texts` must be str or list[str]")

    # ------------------------------------------------------------------
    # 2. Batch-tokenize (no BOS/EOS – we add EOS ourselves)
    # ------------------------------------------------------------------
    enc = tok(
        texts,
        add_special_tokens=False,          # we control EOS ourselves
        padding=False,
        truncation=False,
        return_attention_mask=False,
    )

    # ------------------------------------------------------------------
    # 3. Flatten + append EOS after *each* example
    # ------------------------------------------------------------------
    flat_ids = []
    eos_id = tok.eos_token_id
    for ids in enc["input_ids"]:
        flat_ids.extend(ids)
        if eos_id is not None:
            flat_ids.append(eos_id)        # one EOS per QA pair

    return flat_ids

# -------------------------------------------------
# 2. Pack into fixed-size blocks (no cross-doc bleed)
# -------------------------------------------------
def pack_ids_to_blocks(ids, block_size):
    """
    ids: list[int]
    Returns: Dataset of dicts with `input_ids` and `attention_mask`
    """
    blocks = []
    # Drop the tail that is shorter than block_size
    usable = len(ids) - (len(ids) % block_size)
    ids = ids[:usable]

    for i in range(0, usable, block_size):
        chunk = ids[i: i + block_size]
        blocks.append({
            "input_ids": chunk,
            "attention_mask": [1] * len(chunk),
            # labels = input_ids for causal LM training
            "labels": chunk.copy(),
        })
    return Dataset.from_list(blocks)

# -------------------------------------------------
# 3. Build train / eval splits
# -------------------------------------------------
# raw_texts → list[str]  (your existing list of Q/A strings)
dataset = Dataset.from_list([{"text": t} for t in raw_texts])

train_val = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_val["train"]
eval_dataset  = train_val["test"]

# ------------------------------------------------------------------
# IMPORTANT: convert the column to a *plain Python list* before tokenising
# ------------------------------------------------------------------
train_ids = tokenize_append_eos(list(train_dataset["text"]))
eval_ids  = tokenize_append_eos(list(eval_dataset["text"]))

# Pack
train_dataset = pack_ids_to_blocks(train_ids, BLOCK_SIZE)
eval_dataset  = pack_ids_to_blocks(eval_ids,  BLOCK_SIZE)

# fix to reduce memory usage by reducing eval sequence length
MAX_EVAL_LEN = 256  # even 384 works

def truncate_eval(example):
    return {
        "input_ids": example["input_ids"][:MAX_EVAL_LEN],
        "attention_mask": example["attention_mask"][:MAX_EVAL_LEN],
        "labels": example["labels"][:MAX_EVAL_LEN],
    }

eval_dataset = eval_dataset.map(
    truncate_eval,
    remove_columns=eval_dataset.column_names,
)


# use an even smaller subset size of the original
EVAL_SUBSET_SIZE = 500  # 200–500 is plenty
eval_dataset = eval_dataset.shuffle(seed=42).select(range(EVAL_SUBSET_SIZE))

print(
    f"Prepared train_set len {len(train_dataset)} "
    f"and eval_set len {len(eval_dataset)} "
    f"packed training chunks of {BLOCK_SIZE} tokens ✅"
)


Map:   0%|          | 0/7636 [00:00<?, ? examples/s]

Prepared train_set len 73373 and eval_set len 500 packed training chunks of 1024 tokens ✅


# Init Trainer Params

In [7]:
import os
from transformers import TrainingArguments, EarlyStoppingCallback, Trainer, TrainerCallback
from trl import SFTTrainer

# 1) Absolute, writable, persistent output dir
OUTPUT_DIR = "/storage/models/wtk-gamma-llama3-lora-v9/"

# 2) Build explicit TrainingArguments (NO dict here)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    resume_from_checkpoint=True,  # Fresh start unless resuming
    num_train_epochs=3,  # Allow up to 3, but early stop will kick in
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1, # reduce eval batch size to avoid OOM errors
    gradient_accumulation_steps=4,  # Balances batch size (~8 effective)
    lr_scheduler_type="cosine",  # Keep: stable for esoteric data
    learning_rate=5e-6, 
    warmup_ratio=0.1,  # Adjusted: gradual ramp for stability
    weight_decay=0.03,  # Keep: prevents overfitting
    fp16=False,  # Keep: A4000 supports BF16
    bf16=True,  # Keep: efficient on Ampere
    logging_steps=10,  # Keep: frequent monitoring
    eval_strategy="steps",
    eval_steps=100,                     # every ~400 examples
    save_steps=100,                     # Keep: regular checkpoints
    save_total_limit=2,                 # keep best + final only
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=0.2,  # Lowered: stabilizes expanded modules
    dataloader_num_workers=4,  # Keep: stable on mounted storage
    prediction_loss_only=True, # prevent retention of logits for metrics
)

# Add EarlyStopping: stop if no improvement for 3 evaluations
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,   # Wait 3 eval steps (150 steps total)
    early_stopping_threshold=0.0008  # Optional: min improvement
)


# Callback to avoid eval until epoch 1 has trained
class DelayEvalUntilEpochCallback(TrainerCallback):
    def __init__(self, start_epoch=1):
        self.start_epoch = start_epoch

    def on_step_end(self, args, state, control, **kwargs):
        if state.epoch is None or state.epoch < self.start_epoch:
            control.should_evaluate = False
        return control
    

# helps with OOM
peft_model.config.use_cache = False
peft_model.generation_config.use_cache = False


trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tok,
    callbacks=[early_stopping, DelayEvalUntilEpochCallback(start_epoch=1)],  # ← ADD HERE
)

print(f"Created SFTTrainer ✅")


Created SFTTrainer ✅


/tmp/ipykernel_425/3545438866.py:62: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer._unsloth___init__`. Use `processing_class` instead.
  trainer = Trainer(


# Train using SFTTrainer (new)

In [7]:
# 4) Start training (this might take a while)
trainer.train()
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 73,373 | Num Epochs = 3 | Total steps = 27,516
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 225,443,840 of 70,779,150,336 (0.32% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7febb89054e0>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1618, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python

KeyboardInterrupt: 

# Resume training using SFTTrainer (only use if resuming)

In [7]:
CHECKPOINT_DIR = OUTPUT_DIR + "/" + "checkpoint-1800"

#4) Start training (this might take a while)
print(f"Resuming from checkpoint: {CHECKPOINT_DIR}")
trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

Resuming from checkpoint: /storage/models/wtk-trineday-mini-llama3-70b-lora-v7//checkpoint-1200


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 21,733 | Num Epochs = 2 | Total steps = 5,434
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 112,721,920 of 70,666,428,416 (0.16% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
1300,1.884500,2.150746
1400,1.940900,2.149331
1500,1.910000,2.148968


Training complete ✅
Training results saved ✅


# After training – Merge

In [6]:
from peft import PeftModel

# Your LoRA repo must contain adapter_model.safetensors + adapter_config.json
ADAPTER_ID="/storage/models/wtk-gamma-llama3-lora-v9/"
peft_model = PeftModel.from_pretrained(model, ADAPTER_ID)
print("Loaded LoRA adapter:", ADAPTER_ID)

/opt/conda/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Loaded LoRA adapter: /storage/models/wtk-trineday-mini-llama3-70b-lora-v7/


In [ ]:
# After trainer.train()
final_model = peft_model.merge_and_unload()
final_model.save_pretrained("/workspace/wtk-gamma-llama3-70b-merged-v9")
tokenizer.save_pretrained("/workspace/wtk-gamma-llama3-70b-merged-v9/")

/opt/conda/lib/python3.11/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


# Push LoRa to Huggingface

In [4]:
from huggingface_hub import HfApi, upload_folder

repo_id = "peers-ai/wtk-gamma-llama3-lora-v9"
folder = "/storage/models/wtk-gamma-llama3-lora-v9/"  # contains adapter_config.json & adapter_model.bin

api = HfApi()
# create the repo if it doesn't exist
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)

# upload all files in the folder
upload_folder(
    repo_id=repo_id,
    folder_path=folder,
    repo_type="model",
)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Uploaded to https://huggingface.co/peers-ai/wtk-trineday-mini-merged-v1


# Load model with Unsloth patching